# Monitoring Drift

In this tutorial, we will look at c2 senarios where a model's performance is severely hurt by model drift. 



In [2]:
! pip install -r requirements.txt


[notice] A new release of pip is available: 25.0.1 -> 25.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
if not hasattr(np, 'bool'):
    np.bool = bool
from keras.datasets import mnist
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.decomposition import PCA
from scipy.stats import ks_2samp
from sentence_transformers import SentenceTransformer
import plotly.express as px
import plotly.graph_objects as go

## Senario 1: Recognizing the digit 1

In this senario, you are a data scientist from the fictitious country of Driftistan. You are tasked with creating a model that recognizes from a handwritten drawing of a digit if a 1 is written or not. **It is currently year 2 and the people of Driftistan only use 3 digits** : 0, 1 and 2. 

To simulate this, we will take an extract of the MNIST dataset and train our model on a subset containing the digits 0, 1 and 2. The model used is a simple random forest.

In [40]:
# Load the MNIST database
(train_X, train_y), (test_X, test_y) = mnist.load_data()


figure=px.imshow(train_X[0], color_continuous_scale='gray', title="first digit of the database")
figure.show()



In [ ]:
# This max digit is the maximum digit that the people of Driftistan know
max_digit=2

indices=np.where(train_y<max_digit+1)

X=train_X[indices]/256
X=X.reshape((X.shape[0],X.shape[1]*X.shape[2]))
y=train_y[indices]
y[y==2]=0

(18623, 784)


In [42]:
train_size=1000


train_indices=np.random.choice(X.shape[0], train_size, replace=False)


X=X[train_indices]
y=y[train_indices]



In [43]:
clf = RandomForestClassifier(max_depth=2)
clf.fit(X, y)

RandomForestClassifier(max_depth=2)

In [ ]:
test_size=300

indices=np.where(test_y<max_digit+1)

test=test_X[indices]/256
test=test.reshape((test.shape[0],test.shape[1]*test.shape[2]))
true_y=test_y[indices]
true_y[true_y==2]=0

test_indices=np.random.choice(test.shape[0], test_size, replace=False)

test=test[test_indices]
true_y=true_y[test_indices]

pred_y=clf.predict(test)
accuracy=accuracy_score(true_y, pred_y)
print("baseline accuracy is : ",accuracy)


accuracy on year 2 is :  0.9733333333333334


Over time, the people of Driftistan gradually learn new digits. In year 3, they add the digit 3, in year 4, the digit 4 etc. 
You did not retrain your model since year 2. Let's see how well it performs over time.

In [45]:
years=np.arange(2,10)

Accuracy=[]

for year in years :

    max_digit=year

    indices=np.where(test_y<max_digit+1)
    test=test_X[indices]/256
    test=test.reshape((test.shape[0],test.shape[1]*test.shape[2]))
    true_y=test_y[indices]
    true_y[true_y!=1]=0

    test_indices=np.random.choice(test.shape[0], test_size, replace=False)

    test=test[test_indices]
    true_y=true_y[test_indices]

    pred_y=clf.predict(test)
    accuracy=accuracy_score(true_y, pred_y)
    Accuracy=Accuracy+[accuracy]
    print(f"accuracy on year {year} is : {accuracy}")

figure = px.line({'year':years,'accuracy':Accuracy}, x='year', y='accuracy', title="Decline of accuracy over time")
figure.show()

accuracy on year 2 is : 0.99
accuracy on year 3 is : 0.9566666666666667
accuracy on year 4 is : 0.9233333333333333
accuracy on year 5 is : 0.94
accuracy on year 6 is : 0.9266666666666666
accuracy on year 7 is : 0.95
accuracy on year 8 is : 0.9266666666666666
accuracy on year 9 is : 0.9066666666666666


Question 1.1 : Why is the accuracy declining over time ?

Question 1.2 : What kind of drift can you see here ? Concept drift or data drift ? Please thoroughly justify your answer.

To further analyse this drift, you can calculate the difference between the distribution of the new test data and the distribution of the training data. Here, the Kolmogorov-Smirnov test is used on a reduced version of the MNIST dataset. The image data is reduced to 4 dimensions using PCA, then the k-s test is run for each dimension and the mean of the resulted stats for each dimension is returned. The rejection threshold is set arbitrarily at 8%. If the statistic is above 8%, we can reject the hypothesis that the 2 samples come from the same distribution.

In [46]:
years=np.arange(2,10)

n_components=4
rejection_threshold=0.08

Test=[]

for year in years :

    max_digit=year

    indices=np.where(test_y<max_digit+1)
    test=test_X[indices]/256
    test=test.reshape((test.shape[0],test.shape[1]*test.shape[2]))
    true_y=test_y[indices]
    true_y[true_y!=1]=0

    test_indices=np.random.choice(test.shape[0], test_size, replace=False)

    test=test[test_indices]
    pca = PCA(n_components=n_components)

    combined_data=np.vstack([X, test])
    transformed_data = pca.fit_transform(combined_data)

    X_reduced=transformed_data[:X.shape[0]]
    test_reduced=transformed_data[X.shape[0]:]

    mean_ks_stat=0

    for i in range(n_components):
        ks_stat, p_value = ks_2samp(X_reduced[:,i], test_reduced[:,i])
        mean_ks_stat=mean_ks_stat+ks_stat

    mean_ks_stat=mean_ks_stat/n_components

    Test=Test+[mean_ks_stat]

figure = px.line({'year':years,'k-s test':Test, "threshold":rejection_threshold}, x='year', y='k-s test', title="k-s of test data versus train data")
figure.add_trace(go.Scatter(x=years,y=rejection_threshold*np.ones(years.shape[0]), name = "Rejection threshold"))
figure.show()

Question 1.3 : Interpret this graph. How does the progression of the k-s test data indicate the presence of drift?

## Senario 2: Movie reviews

The people of Driftistan occasionally enjoy watching movies. This year is once again year 2.  A large film studio has gathered online reviews and would like you to create a model that determines whether a review for their recent movie "Fast and Curious : ENSTA Drift". 

In [ ]:

import pandas as pd

splits = {'train': 'train.parquet', 'validation': 'validation.parquet', 'test': 'test.parquet'}
train_df = pd.read_parquet("hf://datasets/cornell-movie-review-data/rotten_tomatoes/" + splits["train"])
test_df=pd.read_parquet("hf://datasets/cornell-movie-review-data/rotten_tomatoes/" + splits["test"])

train_df

,text,label
0,the rock is destined to be the 21st century's ...,1
1,"the gorgeously elaborate continuation of "" the...",1
2,effective but too-tepid biopic,1
3,if you sometimes like to go to the movies to h...,1
4,"emerges as something rare , an issue movie tha...",1
...,...,...
8525,any enjoyment will be hinge from a personal th...,0
8526,if legendary shlockmeister ed wood had ever ma...,0
8527,hardly a nuanced portrait of a young woman's b...,0
8528,"interminably bleak , to say nothing of boring .",0


In [ ]:
print("Loading all-MiniLM-L6-v2 model...")
st_model = SentenceTransformer('all-MiniLM-L6-v2')

Loading all-MiniLM-L6-v2 model...


This time, you are working with natural language inputs, so the first step is to transform these inputs into vectors. For this, we will doc2vec from the gensim library

In [ ]:
test_df

,text,label
0,lovingly photographed in the manner of a golde...,1
1,consistently clever and suspenseful .,1
2,"it's like a "" big chill "" reunion of the baade...",1
3,the story gives ample opportunity for large-sc...,1
4,"red dragon "" never cuts corners .",1
...,...,...
1061,a terrible movie that some people will neverth...,0
1062,there are many definitions of 'time waster' bu...,0
1063,"as it stands , crocodile hunter has the hurrie...",0
1064,the thing looks like a made-for-home-video qui...,0


In [ ]:
output_dir = "./saved_sbert_model"
os.makedirs(output_dir, exist_ok=True)
st_model.save(output_dir)


In [ ]:


X=np.array([st_model.encode(text) for text in train_df["text"]])

y=np.array(train_df['label'])


clf = RandomForestClassifier(max_depth=2)
clf.fit(X, y)

(8530, 384)
(8530,)


RandomForestClassifier(max_depth=2)

In [ ]:
test_size=1000
test_indices=np.random.choice(test_df["text"].shape[0], test_size, replace=False)

test=test_df.iloc[test_indices,:]
true_y=test['label']

test=np.array([st_model.encode(text) for text in test["text"]])
pred_y=clf.predict(test)
accuracy=accuracy_score(true_y, pred_y)
print("baseline accuracy is ",accuracy)

initial accuracy is  0.631


In year 3, a mysterious wizard teaches sarcasm to a handful of Driftistan citizens. Therefore, some new reviews for "Fast and Curious: ENSTA Drift" are sarcastic. Over time, an increasing proportion of reviews will become sarcastic. We simulate this change by taking a portion of reviews and switching their labels. 

In [ ]:
initial_sarcasm_rate=0.05
yearly_sarcasm_increase=0.01

yearly_reviews=200
first_year=2
number_of_years=np.floor(test_df.shape[0]/yearly_reviews)
years=np.arange(first_year,first_year+number_of_years)

test_df = test_df.sample(frac = 1)


year=first_year
first_review=0
sarcasm_rate=initial_sarcasm_rate
Accuracy=[]

for year in years:

    df=test_df.iloc[first_review:first_review+yearly_reviews,:]

    if year>first_year:
        sarcastic_indexes=df.sample(frac=sarcasm_rate).index
        df.loc[sarcastic_indexes,'label']=1-df.loc[sarcastic_indexes,'label']
        sarcasm_rate=sarcasm_rate+yearly_sarcasm_increase
    


    true_y=np.array(df['label'])

    test=np.array([st_model.encode(text) for text in df["text"]])
    pred_y=clf.predict(test)
    accuracy=accuracy_score(true_y, pred_y)
    print("Accuracy for year ",int(year)," is ",accuracy)
    Accuracy=Accuracy+[accuracy]
    
        
figure = px.line({'year':years,'accuracy':Accuracy}, x='year', y='accuracy', title="Decline of accuracy over time")
figure.show()


Accuracy for year  2  is  0.645
Accuracy for year  3  is  0.635
Accuracy for year  4  is  0.615
Accuracy for year  5  is  0.595
Accuracy for year  6  is  0.575


Question 2.1 : Why is the accuracy declining over time ?

Question 2.2 : What kind of drift can you see here ? Concept drift or data drift ? Please thoroughly justify your answer.

Question 2.3 : What would be the result of a k-s test in this case ?